# From REINFORCE to RLOO to GRPO

## 1. REINFORCE → RLOO

### Problem: A reward alone does not tell us how good an answer is relative to other answers

Suppose the prompt is:

`2 + 2 = ?`

The model generates:

`abcdef4`

Reward = 1.

REINFORCE updates the policy using:

$$
g = R \nabla_\theta \log \pi_\theta(y \mid x)
$$

This reinforces the entire sampled completion.

But consider two situations:

- Q1: An easy question where the model almost always succeeds.
- Q2: A difficult question where the model rarely succeeds.

Both successful completions receive reward = 1, but their significance relative to the model's usual performance may be very different.

**Question:** Can we estimate a prompt-specific baseline without training a separate value network?

### Solution: Sample multiple completions for the same prompt

For Q1, suppose we sample three independent completions:

| Completion | Reward |
|---|---:|
| A1: `abcdef4` | 1 |
| A2: `abcdef6` | 0 |
| A3: `abcdef10` | 0 |

The rewards are:

$$
R = [1,0,0]
$$

Instead of using the raw reward, compare each completion against the other completions.

This leads to **REINFORCE Leave-One-Out (RLOO)**.

### Deriving RLOO advantages

For A1, the baseline is the average reward of A2 and A3:

$$
b_1 = \frac{0+0}{2}=0
$$

$$
A_1 = 1-0=1
$$

For A2:

$$
b_2 = \frac{1+0}{2}=0.5
$$

$$
A_2 = 0-0.5=-0.5
$$

For A3:

$$
b_3 = \frac{1+0}{2}=0.5
$$

$$
A_3 = 0-0.5=-0.5
$$

Therefore:

$$
\boxed{A_{\text{RLOO}}=[1,-0.5,-0.5]}
$$

The general formula for a group of $G$ completions is:

$$
\boxed{
A_i^{\text{RLOO}}
=
R_i-\frac{1}{G-1}\sum_{j\ne i}R_j
}
$$

### What changes in the policy gradient?

REINFORCE:

$$
g = R\nabla_\theta\log\pi_\theta(y\mid x)
$$

RLOO:

$$
g = A_{\text{RLOO}}\nabla_\theta\log\pi_\theta(y\mid x)
$$

For our three completions, the group-average gradient estimator becomes:

$$
\begin{aligned}
g_{\text{RLOO}}=\frac13\big[&
\nabla_\theta\log\pi_\theta(A1\mid x)\\
&-0.5\nabla_\theta\log\pi_\theta(A2\mid x)\\
&-0.5\nabla_\theta\log\pi_\theta(A3\mid x)
\big]
\end{aligned}
$$

Interpretation:

- A1: Positive advantage → reinforce the sequence.
- A2: Negative advantage → discourage the sequence.
- A3: Negative advantage → discourage the sequence.

These describe the individual gradient contributions, not guaranteed probability changes after combining all gradients in a shared-parameter model.

**Key improvement:** RLOO provides positive and negative learning signals by comparing completions from the same prompt, without requiring a learned critic.

### Why "Leave One Out"?

Why not simply subtract the average reward of all three completions?

$$
\bar R=\frac{1+0+0}{3}=\frac13
$$

This would give:

$$
R-\bar R=[0.667,-0.333,-0.333]
$$

The problem is that each completion contributes to its own baseline.

For the usual policy-gradient baseline argument, the baseline must be independent of the sampled action being evaluated, conditional on the prompt.

RLOO excludes the current completion and uses only the other independently sampled completions.

Under standard on-policy assumptions, this **preserves the unbiasedness of the policy-gradient estimator**.

There is also a useful algebraic identity:

$$
\boxed{
A_i^{\text{RLOO}}
=
\frac{G}{G-1}(R_i-\bar R)
}
$$

Therefore, directly subtracting the group mean produces advantages proportional to RLOO advantages.

For $G=3$:

$$
A_i^{\text{RLOO}}=\frac32(R_i-\bar R)
$$

This relationship naturally leads to GRPO.

---

## 2. RLOO → GRPO

### Problem: Different prompts can have very different reward scales

RLOO measures how much better or worse each completion is than the other completions.

But what happens when the reward differences vary substantially across prompts?

For example:

- Group 1: $R=[1,0.5,0]$
- Group 2: $R=[0.51,0.50,0.49]$

The ordering is identical, but Group 1 has much larger reward differences.

RLOO preserves those differences.

**Question:** What if we want to compare completions primarily by their relative standing within each group, while normalizing the reward scale?

### Solution: Standardize the group-relative rewards

GRPO (Group Relative Policy Optimization) uses:

$$
\boxed{
A_i^{\text{GRPO}}
=
\frac{R_i-\bar R}{\sigma_R}
}
$$

where:

$$
\bar R=\frac1G\sum_{i=1}^{G}R_i
$$

and, using population standard deviation,

$$
\sigma_R=
\sqrt{
\frac1G\sum_{i=1}^{G}(R_i-\bar R)^2
}
$$

Practical implementations may add a small $\epsilon$ for numerical stability.

### MRE 1: Compare RLOO and GRPO on Q1

$$
R=[1,0,0]
$$

Mean:

$$
\bar R=\frac13
$$

Standard deviation:

$$
\sigma_R=\frac{\sqrt2}{3}\approx0.4714
$$

GRPO advantages:

$$
\boxed{
A_{\text{GRPO}}
=
[1.414,-0.707,-0.707]
}
$$

Comparison:

| Completion | RLOO | GRPO |
|---|---:|---:|
| A1 | +1.000 | +1.414 |
| A2 | −0.500 | −0.707 |
| A3 | −0.500 | −0.707 |

Notice:

$$
A_{\text{GRPO}}=1.414\,A_{\text{RLOO}}
$$

All advantages have been multiplied by the same constant.

**Observation:** Within this group, the relative weighting of the three completions has not changed.

If everything else is held fixed, and we ignore clipping and other loss-normalization differences, the group gradient is simply rescaled.

This example alone does not explain the motivation for GRPO standardization.

We need a better experiment.

### MRE 2: Large vs. small reward differences

Compare:

$$
R_1=[1,0.5,0]
$$

$$
R_2=[0.51,0.50,0.49]
$$

Calculate the advantages:

| Method | Large reward differences | Small reward differences |
|---|---|---|
| RLOO | $[0.75,0,-0.75]$ | $[0.015,0,-0.015]$ |
| GRPO | $[1.225,0,-1.225]$ | $[1.225,0,-1.225]$ |

This reveals the fundamental difference.

**RLOO preserves reward magnitude!!!!!!!**

>The first group's RLOO advantage magnitude is **50 times** larger than the second group's.

**GRPO normalizes reward magnitude within each group.**

Both groups produce the same standardized advantages because their relative reward patterns are identical.

This can prevent groups with large reward spreads from dominating purely because of their reward scale.

However, it introduces a new concern.

### New problem: Is standardization always desirable?

Consider:

$$
R=[0.51,0.50,0.49]
$$

The differences are tiny.

They might represent meaningful quality differences, or they might simply reflect reward **noise**.

Yet GRPO produces:

$$
A_{\text{GRPO}}=[1.225,0,-1.225]
$$

Small differences can therefore become substantial standardized learning signals.

This may be useful when reward scales are inconsistent across prompts.

But it can be undesirable when a small reward spread indicates uncertainty or noise.

**Important:** GRPO standardization is an optimization choice, not a universally better advantage estimator.

This concern will help motivate Dr. GRPO.

---

## 3. GRPO Is More Than Standardized RLOO

So far, we have compared only advantage estimation.

But the full GRPO algorithm also changes the policy optimization objective.

RLOO is fundamentally a REINFORCE-style estimator.

GRPO combines group-relative advantages with PPO-style optimization.

| Component | RLOO | Original GRPO |
|---|---|---|
| Multiple completions per prompt | Yes | Yes |
| Baseline | Leave-one-out mean | Group mean |
| Group standard deviation | No | Yes |
| Learned critic | No | No |
| PPO-style ratio clipping | Not required | Yes |
| Token aggregation | Sequence log-probability sum | Length-normalized token objective |

### PPO-style clipping in GRPO

Suppose A1 has:

$$
A_{\text{GRPO}}=1.414
$$

For one token, suppose the importance ratio is:

$$
\rho_t=
\frac{\pi_\theta(y_t\mid x,y_{<t})}
{\pi_{\text{old}}(y_t\mid x,y_{<t})}
=1.5
$$

With a clipping range of $[0.8,1.2]$, the positive-advantage PPO surrogate contribution is:

$$
\min(1.5\times1.414,\;1.2\times1.414)
$$

$$
=1.6968
$$

The contribution is in the clipped region, where the surrogate gradient with respect to increasing that ratio becomes zero.

This limits the incentive to increase an already-large ratio further.

RLOO's original REINFORCE-style objective does not contain this clipping mechanism.

However, clipping can also be combined with RLOO advantages.

**Therefore, GRPO is not simply a mathematically necessary upgrade from RLOO.**

It is a particular combination of:

1. Group-relative advantage estimation.
2. Group standard deviation normalization.
3. PPO-style clipped policy optimization.
4. A particular token-level loss aggregation.

---

## 4. The Remaining Credit Assignment Problem

Recall the original completion:

`abcdef4`

Reward = 1.

For an autoregressive model:

$$
\log\pi_\theta(y\mid x)
=
\sum_{t=1}^{T}
\log\pi_\theta(y_t\mid x,y_{<t})
$$

RLOO and GRPO still assign one sequence-level advantage to the generated tokens of that completion.

For example:

$$
g_{\text{RLOO}}
=
A_{\text{RLOO}}
\sum_{t=1}^{T}
\nabla_\theta\log\pi_\theta(y_t\mid x,y_{<t})
$$

The same advantage multiplies the log-probability gradient of `a`, `b`, `c`, `d`, `e`, `f`, and `4`.

Neither method directly identifies which tokens were responsible for the correct answer.

Therefore, **group-relative advantages do not solve token-level credit assignment**.

The methods improve how the sequence-level learning signal is estimated and applied, but they do not automatically discover that the final `4` is the crucial part of `abcdef4`.

---

## 5. Summary: What Changed?

| Algorithm | Central idea | Problem addressed |
|---|---|---|
| REINFORCE | Weight sampled log-probability gradients by reward | Learn from sampled outcomes |
| RLOO | Compare each completion with other completions from the same prompt | Estimate a baseline without a learned critic |
| GRPO | Standardize group-relative advantages and use PPO-style optimization | Normalize reward scales and constrain policy updates |

The conceptual progression is:

**REINFORCE**

"I generated `abcdef4` and received reward = 1. Reinforce that sequence."

↓

**RLOO**

"I generated three answers. `abcdef4` performed better than the other two. Reinforce it and discourage the worse answers."

↓

**GRPO**

"I still compare answers within a prompt, but I normalize their relative reward differences and constrain policy updates with a PPO-style objective."

## 6. Questions Leading to Dr. GRPO

Two important problems remain.

### Question A: Group standard deviation

If the standard deviation becomes very small:

$$
A_i^{\text{GRPO}}
=
\frac{R_i-\bar R}{\sigma_R}
$$

What happens to the learning signal?

What if all rewards are identical, so that:

$$
\sigma_R=0
$$

Could standardization change the relative importance of different prompts?

### Question B: Response length

Consider two groups:

| Group | Correct completion length | Incorrect completion lengths |
|---|---:|---|
| Q1 | 10 tokens | 30, 100 |
| Q2 | 500 tokens | 50, 200 |

The original GRPO objective averages token-level contributions within each completion using a factor of $1/T_i$.

But why should the number of generated tokens affect the learning signal if the reward only evaluates the final answer?

How does length normalization change the relative gradient contribution of short and long completions?

These questions lead naturally to **Dr. GRPO**, which examines biases introduced by reward standard deviation normalization and response-length normalization.

---

**Core takeaway:** RLOO changes how we construct the baseline. GRPO changes both the scale of the group-relative advantage and the policy optimization procedure. Neither is automatically superior in every setting; their differences become meaningful when reward distributions, response lengths, and policy updates vary.